# 🛡️ Kaggle 24/7 Full Web Platform + Single-File GGUF Loader (Zero /tmp Disk Saturation)
Optimisation disque stricte avec redirection TMPDIR, téléchargement direct du fichier GGUF 7 Go sans clonage LFS lourd et 5 optimisations matérielles.

In [ ]:
# 1. Redirection TMPDIR & Nettoyage préventif du disque temporaire /tmp
import os, shutil

os.environ['TMPDIR'] = '/kaggle/working/tmp'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/tmp/pip'
os.makedirs('/kaggle/working/tmp', exist_ok=True)

!rm -rf /tmp/* /kaggle/working/tmp/* /root/.cache/pip || true
print('🟢 Disque temporaire /tmp réinitialisé avec succès !')

In [ ]:
# 2. Clonage résilient du dépôt Git OSINT & Installation avec pré-compilation (Zero Build Overhead)
import time, subprocess

!rm -rf /kaggle/working/projet_osint

clone_url = 'https://github.com/your-repo/projet_osint.git'
cloned = False

for attempt in range(1, 6):
    print(f'Tentative de clonage Git ({attempt}/5)...')
    res = subprocess.run(['git', 'clone', clone_url, '/kaggle/working/projet_osint'])
    if res.returncode == 0:
        cloned = True
        print('🟢 Dépôt Git OSINT cloné avec succès !')
        break
    time.sleep(3)

if not cloned:
    raise RuntimeError('Échec du clonage Git OSINT.')

!pip install --no-cache-dir --prefer-binary huggingface_hub "llama-cpp-python[server]"

In [ ]:
# 3. Téléchargement direct du FICHIER UNIQUE GGUF & Démarrage de llama-server
import os, sys, subprocess, time, glob
from huggingface_hub import list_repo_files, hf_hub_download

repo_id = 'KevinJK51/Qwen3.6-12B-IQ-Ultra-Heretic-Uncensored-Thinking-V2-Hightop-GGUF'
model_dir = '/kaggle/working/models'
os.makedirs(model_dir, exist_ok=True)

print(f'🔍 Recherche du fichier GGUF unique dans {repo_id}...')
files = list_repo_files(repo_id)
gguf_files = [f for f in files if f.endswith('.gguf')]

if not gguf_files:
    raise RuntimeError('Aucun fichier GGUF trouvé.')

# Sélection du fichier IQ3/IQ4 ou premier fichier GGUF
selected_file = next((f for f in gguf_files if 'iq4' in f.lower() or 'iq3' in f.lower() or 'q4' in f.lower()), gguf_files[0])
print(f'⬇️ Téléchargement direct du fichier unique sans historique Git : {selected_file}...')

model_path = hf_hub_download(repo_id=repo_id, filename=selected_file, local_dir=model_dir)
print(f'🟢 Poids GGUF installés avec succès (0 saturation /tmp) : {model_path}')

print('🚀 Lancement de llama-server avec les 5 optimisations matérielles...')
llama_cmd = [
    sys.executable, '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8080',
    '--n_ctx', '32768',
    '--n_threads', '2'
]
llama_process = subprocess.Popen(llama_cmd)
time.sleep(10)

In [ ]:
# 4. Démarrage de FastAPI & Tunnel HTTPS Cloudflare (Avec sys.executable & sys.path sécurisés)
import os, sys, subprocess, time, re, shutil

backend_dir = '/kaggle/working/projet_osint/backend'
os.chdir(backend_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

!pip install --no-cache-dir -r requirements.txt
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared || true

print('Démarrage du Serveur FastAPI (Frontend + Backend) sur port 8000...')
server_process = subprocess.Popen([sys.executable, '-m', 'app.main'], cwd=backend_dir)
time.sleep(6)

print('Lancement du Tunnel HTTPS...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(25):
    line = tunnel_process.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n======================================================')
            print(f'🚀 VOTRE INTERFACE WEB EST EN LIGNE H24 : {match.group(0)}')
            print('======================================================\n')
            break
    time.sleep(1)

In [ ]:
# 5. Boucle d'exécution continue 24/7 avec Checkpoints SQLite horaires et sys.path sécurisé
import time, os, sys

backend_dir = '/kaggle/working/projet_osint/backend'
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)
os.chdir(backend_dir)

from app.cloud_sync.kaggle_persistence import KagglePersistenceManager
from app.cloud_sync.garbage_collector import GarbageCollectorManager

print('🟢 Boucle d\'exécution continue 24/7 active...')
for hour in range(1, 11):
    time.sleep(3600)
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] Checkpoint State Heure {hour}/10...')
    GarbageCollectorManager.cleanup_temp_storage()
    KagglePersistenceManager.checkpoint_state()